## Resting-state ECG: Import and Preprocessing

##### ECG preprocessing pipeline for PREACT-digital  

Details:
* 2x/day, 30 sec during active EMA assessment (T0, T20, TPost), while each active assessment includes max. 16 days
  
* sampling rate: 300 Hz (i.e., 300 samples/1 sec => one sample every ~3.33 ms)
  
* 30 sec -> 300 x 30 = 9000 data points per ecg session

* the raw data export provides values in V:
    - 0.001 V ($10^{-3}$) = 1 mV
    - 0.000001 V ($10^{-6}$) = 1 µV microvolt
    - example: 0.000012 V ($10^{-5}$) = 0.012 mV = 12 µV


Pipeline (wip):

1. **Import data:** Variation in data format   
    - raw data as StringValues 'ECGVoltageOffset' with [t, v] from 14.03.2023 - 20.11.2023 (one string per ecg session -> data need to be extracted)  
    - raw data as single '.csv' files [t, v] per ecg session from 17.11.2023 - today

2. **Concatenate:** separate 30s snippets per participant  -> this is not implemented so far

3. **Preprocessing & HRV computation:** NeuroKit  
    - Clean signal (remove noise and artifacts) & bandpass filtering  
    - R-Peak detection    
    - RR Interval computation   
    - HRV Metric (e.g. time-domain metric: RMSSD (ms)) based on 30 sec assessment

4. **Run Quality Control (QC):** generate a summary table (value range: min, max, mean, sd; sample size; range of missings in time series)  

5. **Export:** ecg features as `.parquet` file   
       

##### ECG Amplitudes not clear defined for wearables but can be lower compared to lab-assessments

| ECG Component     | Range (mV) | 
|------------------|-------------------|
| Baseline noise   | 0.005 – 0.05 mV   | 
| P wave           | 0.05 – 0.2 mV     | 
| QRS complex      | 0.3 – 1.5 mV      | 
| T wave           | 0.05 – 0.3 mV     | 


##### Formula to convert V in mV (1 V are 1000 mV): 

$$
\text{V} \times 1000 = \text{mV}
$$

example:

$$
0.000012\ \text{V} \times 1000 = 0.012\ \text{mV}
$$

For data preprocessing and HRV computation the neurophysiological toolkit [Neurokit2](https://neuropsychology.github.io/NeuroKit/functions/ecg.html#) was used ([Makowski et al. 2021](https://link.springer.com/article/10.3758/s13428-020-01516-y)). See also [Pham et al. 2021](https://www.mdpi.com/1424-8220/21/12/3998) for a step-by-step guide for HRV analysis using Neurokit2.

-----------------------

#### TO-DOES (for Michal):

Ideally work through the TO DO's in the proposed order:

* ecg data from November 2023 until today:
    - loading all .csv files from 17.11.2023 - until today takes hours if you just concatenat them. Instead the idea was to create batches. however, I created 3-moth batches with sub-batches and tried to concatenate these but this is also too slow because each single file contains 9000 rows and we have 676686 files with 9000 rows each (this is larger than the batches we concatenate in the data_load script with the very first passive data backup). Please find out if this is really the best way in terms of memory and adjust it if you find something more feasible and faster

    - some rows in the .csv files have more than 9000 data points (72000 means 8x 9000, 54000 means 6x 9000, 45000 means 5x 9000, 27000 means 3x 9000, 18000 means 2x 9000 -> after inspecting single files, it seems so that some data are stored more than once but it is not completely clear (needs more investigation). every single file should only contain 9000 data points!

    - the latest .csv file is from 30.09.2025 -> either this is a code bug OR an export bug from thrive - please double check (some 2026 folders seem to include only Nokia_300.csv files with timestamps from 2025 and 2024)

* Concatenate the cleaned raw data from March - Nov 2023 with the once from Nov23 - today

* Double check and potentially adjust timezone if necessary (I didn't checked that to date)

* Run the preproc loop for all data and plot the violin plot to investigate data loss for RMSSD (HRV metric)

* If it's high overall, go back into the preproc loop and investigate which parameters you could adjust and play around (ideally based on literature evidence so we can cite and argue why we chose which parameter in which size)

* When you have a final version add the following columns to the data frame or make sure they are already included: ID, FOR-ID, timestamp, measurement_burst, ecg_session_nr (e.g. b1_d1_ecg_ses1, b1_d1_ecg_ses2), ema ecg control item

* export as .parquet


Further TO-DO's that can be tackled afterwards:
* probably a few assessments will get lost by that because they are not directly followed by an active assessment (ema) but just activated randomly -> do we want to keep ecg sessions out of beep activation and flag them as 'ecg out of beep activation' vs 'triggered by beep' in a separate column?
* participants can start the ecg more than once after the beep tells them to start ecg -> if more than one session is available for hrv computation for a beep how to decide which one to keep?
* compute additional features? (NeuroKit2 enables to infer more)

---

In [ ]:
from pyprojroot import here # define relative paths to the project root (working directory)
from pathlib import Path
from collections import defaultdict
import sys
from datetime import date
import gc  
import os
import re
import glob
import pickle

import numpy as np
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------
#  Define paths
# ------------------------------------

# .here is located as invisible file in the project root working directory
PROJECT_ROOT = here() 

# add project root to sys.path
sys.path.append(str(here())) 

# import local paths from env.py
from env import raw_path, ecg_path, backup_path, preprocessed_path 



#### 1. Import

Variation in ECG data formats during data collection requires two separate data import steps indicated as 1.1 and 1.2 followed by harmonization into a single format.

#### 1.1 Import raw data as StringValues from 14.03.2023 - 20.11.2023  

**Period:** 14 March 2023 until 20 November 2023  
**Description:** ECG data are stored as stringValue   
* (one string (= 1 row) contains 300 samples, i.e. all samples for 1 sec)   
* located within the overall passive datase  
* type 'RawECGVoltage'  


path: `/sc-projects/sc-proj-cc15-preact/SP6/backup/first_backup/tiki_backup_2023-03-14_1.csv ...` [latest Update 2025-09-21]


In [ ]:
# define the pattern for passive data files
file_pattern = os.path.join(backup_path, 'first_backup/tiki_backup_*.csv')

# use glob to find all matching files
file_list = glob.glob(file_pattern)

# sort the file list for consistent ordering
file_list.sort()

In [ ]:
# define the dtype for columns that are known to be problematic
dtype_spec = {
    'startTimestamp': 'str',  # load as string initially
    'endTimestamp': 'str'     # load as string initially
}

# create a list to hold all the dataframes
all_dfs = []

# loop over the files and read them with date parsing
for filename in file_list:
    df = pd.read_csv(filename, dtype=dtype_spec, low_memory=False) # low-memory=False: read all files at once, then infer types (more reliable, avoids risk of inaccurate data types but uses more memory)

    # parse timestamps for THIS file
    for col in ["startTimestamp", "endTimestamp"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')

    # derive date & index from THIS filename
    m = re.match(r'tiki_backup_(\d{4}-\d{2}-\d{2})_(\d+)\.csv', os.path.basename(filename))
    if m:
        df['date'] = m.group(1)
        df['index'] = int(m.group(2))

    all_dfs.append(df)

# concatenate all dataframes
df_1 = pd.concat(all_dfs, ignore_index=True)

# optionally, drop the 'date' and 'index' columns if they are no longer needed
df_1.drop(columns=['date', 'index'], inplace=True)


In [ ]:
df_1.head() 

In [ ]:
# sanity check: all IDs included? CHECK
len(df_1["customer"].unique())

In [ ]:
# sanity check: entire period included? CHECK 
min_starttimestamp = df_1['startTimestamp'].min()
max_endtimestamp = df_1['endTimestamp'].max()
print(min_starttimestamp)
print(max_endtimestamp)

# 2023-03-14 13:13:20+00:00
# 2023-11-21 08:05:00+00:00

In [ ]:
# quick sanity check
'stringValue' in df_1.columns

In [ ]:
# datetime is still UTC+01:00 
df_1.dtypes

In [ ]:
# adjust datetime to UTC
for col in ["startTimestamp", "endTimestamp"]:
    df_1[col] = (
        pd.to_datetime(df_1[col], utc=True, errors="coerce", unit="ms")
    )

# extract date and hour part
df_1["start_day"] = df_1.startTimestamp.dt.normalize()
df_1["start_hour"] = df_1.startTimestamp.dt.hour

# extract customer identifier
df_1["customer"] = df_1.customer.str.split("@").str.get(0)
df_1["customer"] = df_1["customer"].str[:4]


In [ ]:
# rename columns
df_1.rename(columns = {"customer": "id",
                     "type": "modality",
                     "startTimestamp": "timestamp_start",
                     "endTimestamp": "timestamp_end"},
                       inplace=True)

# only keep relevant variables
df_1 = df_1[['id', 'timestamp_start', 'timestamp_end', 'timezoneOffset', 'modality',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]


In [ ]:
# filter ecg data
df_1_filtered = df_1[df_1["modality"] == "RawECGVoltage"].copy()

df_1_filtered.head(3)

In [ ]:
# very first ECG session is April 27 2023 
df_1_filtered['timestamp_start'].min()

In [ ]:
# total unique IDs
print("All modalities:", df_1["id"].nunique())

# only ECG
print("ECG:", df_1_filtered[df_1_filtered["modality"] == "RawECGVoltage"]["id"].nunique())

print("--------------------")

# unique IDs by each modality
#print(df_1.groupby("modality")["id"].nunique())


#### ECGVoltageOffset

* resting-state ECG data are stored as stringValue for this time period
* convert stringValue in two separate variables:
  * `t` = relative time [sec] within recording window 
  * `v` = microvoltage (electrical potential difference between two electrodes) -> changes in voltage over time (amplitude of the electrical signal at a given moment)
 
One string (= 1 row) contains 300 samples, i.e. all samples for 1 sec)

In [ ]:
# convert stringValues in two separte variables: t and v
import json
import re

def extract_t_v_lists(json_str):
    try:
        # replace decimal commas in numbers only (match digits with commas)
        fixed_str = re.sub(r'(\d),(\d)', r'\1.\2', json_str)
        parsed = json.loads(fixed_str)
        ecg_data = parsed.get("ECGVoltageOffset", [])
        t_list = [float(entry["t"]) for entry in ecg_data]
        v_list = [float(entry["v"]) for entry in ecg_data]
        return pd.Series([t_list, v_list])
    except Exception as e:
        print("Failed to parse row:", e)
        return pd.Series([None, None])

# apply to the DataFrame
df_1_filtered[["t", "v"]] = df_1_filtered["stringValue"].apply(extract_t_v_lists)

# sanity check: ensure lists are the same length in both columns
assert all(df_1_filtered["t"].str.len() == df_1_filtered["v"].str.len()), "Mismatch in t and v list lengths"


In [ ]:
# take each list in the specified columns (ecg_relative_t, ecg_v) and expand them so that each item in the list becomes its own row
df_1_ecg = df_1_filtered.explode(["t", "v"]).reset_index(drop=True)

# sanity check: 300 samples per second, i.e. 301 should be the next sec in timestamp_start -> YES
df_1_ecg.head(301)

In [ ]:
# sanity check (all IDs included)
df_1_ecg["id"].nunique() 

In [ ]:
# general plausibility check before preprocessing (values in V)
print("Max:", df_1_ecg['v'].max())
print("Min:", df_1_ecg['v'].min())

# typical QRS complex (= one heartbeat) for a scanwatch is: 0.3 - 1.5 mV
# Max/Min values are quite high/low (possible noise/artefacts)

In [ ]:
# general plausibility check: row comparison before stringValue extraction vs. after
print("df_1_filtered shape:", df_1_filtered.shape)
print("df_1_ecg shape:", df_1_ecg.shape)

In [ ]:
# remove unnecessary columns
df_1_ecg = df_1_ecg.drop(columns=["stringValue", "booleanValue","doubleValue","longValue"])

In [ ]:
# final data frame
df_1_ecg.head()

#### 1.2 Import raw data as single .csv files [t, v] from 17.11.2023 - today

**Period:** 17 November 2023 until TODAY   
**Description:** ECG data are stored in `ID_yyyy-mm-dd_hh:mm:ss_Nokia_300.csv`files including a t and a v column (one .csv file per ecg session) 

<u>regex pattern:</u>

* **Group 1:** Customer ID -> `0AlyaEtHo8K3Cyo8`
* **Group 2:** Date -> `2024-04-08`
* **Group 3:** Time -> `12:04:46`

path: `/sc-projects/sc-proj-cc15-preact/SP6/raw/tiki_backup_files/export_tiki_daymonthyear ...` 

In [ ]:
# define the pattern for passive data files
file_pattern = os.path.join(raw_path, "tiki_backup_files", "export_tiki_*", "*_Nokia_300.csv")

# use glob to find all matching files
file_list = glob.glob(file_pattern, recursive=True)

# sort the file list for consistent ordering
file_list.sort()

# sanity check
print(len(file_list), "Nokia_300.csv files found")

In [ ]:
# sanity check: latest recorded ecg session (latest timestamp is 2025-09-30) 
# where are the data between October 2025 and April 2026? again a change in data format by thrive? 
# At least some 2026 folder seem to include no Nokia_300.csv files with a 2026 timestamp

from datetime import datetime

def extract_ts_fast(filename):
    parts = Path(filename).stem.split("_")
    return datetime.strptime(
        f"{parts[1]}_{parts[2]}",
        "%Y-%m-%d_%H:%M:%S"
    )

latest_file = max(file_list, key=extract_ts_fast)

print("Latest file:", latest_file)
print("Timestamp:", extract_ts_fast(latest_file))

------------

#### THIS BATCH IMPORT IS TOO SLOW: NEED TO BE ADJUSTED TO APPLY PREPROC 

Issue: loading all .csv files from 17.11.2023 - until today takes hours if you just concatenat them. Instead the idea was to create 3 month batches (due to the fact that each single file contains 9000 rows). however, these batches are also very slow because we have 676686 files with 9000 rows.
* Please find out if this is really the best way in terms of memory and adjust it if you find something more feasible and faster

Procedure so far:
* split data into 3-month batches
* split each 3-month batch (e.g. quarter) into smaller chunks (sub-batches) to reduce too many files in memory at once

In [ ]:
# define the pattern for passive data files
file_pattern = os.path.join(raw_path, "tiki_backup_files", "export_tiki_*","*_Nokia_300.csv")

dtype_spec = {
    "startTimestamp": "string",
    "endTimestamp": "string"
}

sub_batch_size = 2000  # smaller chunks inside each quarter


# helper: parse metadata (ID and timestamp) from filename to add it as extra columns (the .csv files do not include these medata so far)
def parse_filename_metadata(filename):
    """
    Example filename:
    08d669BLAapfABc7_2024-09-04_12:08:06_Nokia_300.csv
    """
    stem = Path(filename).stem
    parts = stem.split("_")

    if len(parts) < 5:
        raise ValueError(f"Unexpected filename format: {filename}")

    file_id = parts[0][:4]
    timestamp_start = pd.to_datetime(
        f"{parts[1]}_{parts[2]}",
        format="%Y-%m-%d_%H:%M:%S",
        errors="coerce"
    )

    return file_id, timestamp_start


# quarterly batches
def get_quarter_label(ts):
    if pd.isna(ts):
        return "unknown"
    quarter = (ts.month - 1) // 3 + 1
    return f"{ts.year}_Q{quarter}"


# find all files
file_list = sorted(glob.glob(file_pattern))
print(f"{len(file_list)} Nokia_300.csv files found")

if not file_list:
    raise FileNotFoundError("No matching Nokia_300.csv files found.")


# group files into quarters
files_by_quarter = defaultdict(list)

for filename in file_list:
    try:
        _, ts = parse_filename_metadata(filename)
        batch_label = get_quarter_label(ts)
        files_by_quarter[batch_label].append(filename)
    except Exception as e:
        print(f"Skipping file due to filename parse error: {filename} | {e}")

quarter_keys = sorted(files_by_quarter.keys())

print("\nFiles per 3-month batch:")
for key in quarter_keys:
    print(f"{key}: {len(files_by_quarter[key])} files")


# helper: read one file and add metadata columns
def read_one_file(filename):
    df = pd.read_csv(
        filename,
        dtype=dtype_spec,
        low_memory=False   # use default C engine
    )

    file_id, timestamp_start = parse_filename_metadata(filename)

    df["id"] = file_id
    df["timestamp_start"] = timestamp_start

    for col in ["startTimestamp", "endTimestamp"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    return df


# read each quarter into a dataframe using sub-batches
batch_dataframes = {}

for batch_label in quarter_keys:
    current_files = files_by_quarter[batch_label]
    print(f"\nProcessing batch {batch_label} with {len(current_files)} files")

    quarter_parts = []

    for start_idx in range(0, len(current_files), sub_batch_size):
        sub_files = current_files[start_idx:start_idx + sub_batch_size]
        dfs = []

        print(
            f"  Sub-batch {start_idx} to "
            f"{min(start_idx + sub_batch_size - 1, len(current_files) - 1)}"
        )

        for i, filename in enumerate(sub_files, start=1):
            try:
                df = read_one_file(filename)
                dfs.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")

        if dfs:
            sub_batch_df = pd.concat(dfs, ignore_index=True)
            quarter_parts.append(sub_batch_df)

            print(f"    Sub-batch shape: {sub_batch_df.shape}")

            del dfs
            gc.collect()

    if quarter_parts:
        batch_df = pd.concat(quarter_parts, ignore_index=True)
        batch_dataframes[batch_label] = batch_df
        print(f"Stored {batch_label} as DataFrame with shape {batch_df.shape}")

        del quarter_parts
        gc.collect()
    else:
        print(f"No readable files in batch {batch_label}")

----------

#### 2. Concatenate data from format 1.1 and 1.2

#### 3. PREPROCESSING & HRV computation

Use 2500 .csv files as starting point to write the preprocessing pipeline (adjust this part as soon you have all ecg sessions concatenated in one large data frame): loop over all files and preprocess every session separately

PREPOC LIBRARY: NeuroKit2

* There are several different established libraries for ecg preprocessing. after trying out different libraries I decided to stick on [NeuroKit2](https://neuropsychology.github.io/NeuroKit/functions/ecg.html) 

* Reason: 
    - In comparison to general signal processing libraries such as [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.datasets.electrocardiogram.html) NeuroKit2 is specifically designed for physiological signals like ECG or PPG. It incorporates built-in knowledge of ECG morphology (e.g., P waves, QRS complexes, and T waves). As a result, many functions already include domain-specific processing steps, so users do not need to implement everything from scratch. Furthermore, NeuroKit2 is well established in the literature, making it a reliable choice for standardized ECG preprocessing.


   - If, however, you want to develop custom signal processing pipelines or implement algorithms from first principles, SciPy is more suitable. This approach requires a deeper understanding of signal processing methods and how different algorithms affect the data

* Memo: I removed all visual plots to keep the preproc as clean and short as possible. we can re-add them later after fixing the import section

In [ ]:
# OPTION 1

# for testing, use only the first 2500 files (= first 2500 ecg sessions)
# later try out parallel processing with e.g. joblib to apply the preprocessing to the whole data set 
# and turn the file-wise processing in a function

import time

rows = []
times = []

for i in range(2500):

    # starttime for this file to check performance
    start = time.perf_counter()

    df_ecg_tmp = pd.read_csv(file_list[i], low_memory=False)
    
    # keep at most the first 9000 rows
    # df_ecg_tmp = df_ecg_tmp.iloc[:9000]

    # convert ecg from volts (v) to millivolts (mV)
    df_ecg_tmp["mV"] = df_ecg_tmp["v"] * 1000
    
    fs = 300 # sampling frequency in Hz
    method = "neurokit"  # R-peak detection method
    
    # convert raw ecg signal to numpy array (NeuroKit2 only takes np.arrays as input)
    ecg_arr = df_ecg_tmp["mV"].to_numpy()

    # clip extreme values to reduce noise/ artifacts
    ecg_clipped = np.clip(ecg_arr, -2, 2)  # arbitrarily chosen; note: spikes larger than 2 mV are not common (at least in lab assessments)
    
    # preprocess the ECG signal using NeuroKit2: https://neuropsychology.github.io/NeuroKit/functions/ecg.html
    # this function runs different preprocessing steps in parallel (instead of running them separately): 
    # cleaning, peak detection, hr computation, signal quality assessment, 
    # QRS complex delineation, cardiac phase determination (the return parameters include all these informations)
    signals, info = nk.ecg_process(ecg_clipped, sampling_rate=fs, method=method)

    rpeaks = info["ECG_R_Peaks"]
    rr_ms = np.diff(rpeaks) / fs * 1000

    # HRV time-domain features (you can also do frequency/nonlinear) // Disable plots
    hrv = nk.hrv_time(info["ECG_R_Peaks"], sampling_rate=fs, show=False)
    hrv_rmssd = hrv["HRV_RMSSD"].iloc[0]

    rows.append({
        "index": i,
        "Filename Path": file_list[i],
        "Number of Rows": len(df_ecg_tmp),
        "low (mV)": df_ecg_tmp["mV"].min(),
        "max (mV)": df_ecg_tmp["mV"].max(),
        "RR min": rr_ms.min(),
        "RR max": rr_ms.max(),
        "RR mean": rr_ms.mean(),
        "HRV_RMSSD": hrv_rmssd
        
    }
               )
    times.append(time.perf_counter() - start)
    print(i, time.perf_counter()-start)


df = pd.DataFrame(rows)

print(df)
print("Avg runtime:", sum(times) / len(times))
print("Max runtime:", max(times))


In [ ]:
# OPTION 2
# an valid alternative to the function nk.ecg_preproc is to do each step separately as described below 

import time

rows = []
times = []

for i in range(2500):

    # starttime for this file to check performance
    start = time.perf_counter()

    df_ecg_tmp = pd.read_csv(file_list[i], low_memory=False)
    
    # keep at most the first 9000 rows
    # df_ecg_tmp = df_ecg_tmp.iloc[:9000]

    # convert ecg from volts (v) to millivolts (mV)
    df_ecg_tmp["mV"] = df_ecg_tmp["v"] * 1000
    
    fs = 300 # sampling frequency in Hz
    method = "neurokit"  # R-peak detection method
    
    # convert raw ecg signal to numpy array 
    ecg_arr = df_ecg_tmp["mV"].to_numpy()

    # clip extreme values to reduce noise/ artifacts
    ecg_clipped = np.clip(ecg_arr, -2, 2)  # arbitrarily chosen; note: spikes larger than 2 mV are not common (at least in lab assessments)
    
    # clean 
    ecg_cleaned = nk.ecg_clean(ecg_clipped, sampling_rate=fs, method="neurokit")

    # detect peaks
    _, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=fs, correct_artifacts=True)

    # extract peaks
    rpeaks = info["ECG_R_Peaks"]

    # check if enough peaks exist
    if len(rpeaks) < 3:
            raise ValueError("Too few R-peaks")
    
    # compute RR intervals
    rr_ms = np.diff(rpeaks) / fs * 1000

    if len(rr_ms) == 0:
            raise ValueError("Empty RR intervals")
    
    # HRV time-domain feature computation
    hrv = nk.hrv_time(rpeaks, sampling_rate=fs, show=False)
    hrv_rmssd = hrv["HRV_RMSSD"].iloc[0]

    rows.append({
        "index": i,
        "Filename Path": file_list[i],
        "Number of Rows": len(df_ecg_tmp),
        "low (mV)": df_ecg_tmp["mV"].min(),
        "max (mV)": df_ecg_tmp["mV"].max(),
        "RR min": rr_ms.min(),
        "RR max": rr_ms.max(),
        "RR mean": rr_ms.mean(),
        "HRV_RMSSD": hrv_rmssd
        
     }

)
    times.append(time.perf_counter() - start)
    print(i, time.perf_counter()-start)


df = pd.DataFrame(rows)

print(df)
print("Avg runtime:", sum(times) / len(times))
print("Max runtime:", max(times))



Explore if RMSSD (time-domain HRV metric) is in a plausible range
* In general, what is considered low or high depends heavily on the individual's baseline (higher: better, lower: worse)
* but a very broad orientation can be this:

| RMSSD (ms)     | Interpretation | 
|------------------|-------------------|
| < 20 ms    | low (stress, fatigue, illness) |  
| 20 - 40 ms          | below average     | 
| 40 - 80 ms     | normal / healthy      | 
| 80 - 120 ms         | high (good recovery)     | 
| > 120 ms         | very high (often athletes)     | 


In [ ]:
import plotly.express as px

rmssd = df["HRV_RMSSD"].dropna()

# apply both lower and upper bounds
rmssd_clean = rmssd[(rmssd >= 5) & (rmssd <= 200)]

# count excluded values (outside this range)
excluded_count = ((rmssd < 5) | (rmssd > 200)).sum()
total_count = len(rmssd)
excluded_pct = excluded_count / total_count * 100

excluded_pct = excluded_count / total_count * 100

# filtering (cut RMSSD values >= 200)
rmssd_clean = rmssd[rmssd < 200]
excluded_count = (rmssd >= 200).sum()
total_count = len(rmssd)

# violin plot (horizontal)
fig = px.violin(
    x=rmssd_clean,
    box=True,
    points="all",
    title=f"RMSSD Violin Plot (5 - 200 ms) | n sessions ={len(rmssd)}  | Excluded={excluded_count}/{total_count} ({excluded_pct:.1f}%)"
)

fig.update_traces(
    orientation="h",   # horizontal
    meanline_visible=True
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="HRV_RMSSD (ms)",
    yaxis_title="",
)

fig.show()

a typical clean data set should look like
* peak around 30 - 60 ms
* right-skewed tail
* very few values > 150 ms


* even if it strongly depends on age, health status, and preprocessing conditions in general it's arguable to define
  - too low: RMSSD < 5 - 10 ms
  - too large: RMSSD > 150 - 200 ms

Thus: 5 - 200 ms as acceptable range is quite a liberal choice with still 20.5% data loss in 2500 ecg sessions

In [ ]:
low_outliers = (rmssd < 5).sum()
high_outliers = (rmssd > 200).sum()

print(f"Low (<5): {low_outliers}")
print(f"High (>=200): {high_outliers}")

In [ ]:
# show all metrics (we can infer more from NeuroKit but I reduced it for know)
df.head(50)

# still an issue: some rows in the .csv files have more than 9000 data points 
# 72000 means 8x 9000, 54000 means 6x 9000, 45000 means 5x 9000, 27000 means 3x 9000, 18000 means 2x 9000
# after inspecting single files, it seems so that some data are stored more than once but it is not completely clear (needs more investigation), 
# every single file should only contain 9000 data points 

In [ ]:
# note: 
# removed all single case visualizations of preprocessing steps (raw, clean, peak detection etc.) to make the notebook as clean as possible for know
# (can add them later again if you want)

